
# 06 — Article-Ready Tables and Figures

Notebook ini merupakan tahap **konsolidasi final**, bukan eksperimen baru.
Seluruh angka ditarik dari output tervalidasi Notebook 01–05 dan disusun menjadi:

1. tabel utama artikel;
2. tabel suplemen;
3. figur utama beresolusi 300 dpi;
4. manifest caption dan penempatan;
5. *results ledger* untuk menjaga konsistensi angka dan terminologi;
6. checklist kesiapan penulisan manuskrip.

## Prinsip pelaporan

- `graduate output intensity` bukan *graduation rate* cohort;
- `graduate-entry flow intensity` bukan *completion rate*;
- seluruh hubungan bersifat agregat, ekologis, dan cross-sectional;
- kata **associated with** digunakan, bukan **caused by**;
- hasil prediktif utama menggunakan validasi provinsi-terpisah;
- tabel utama dibatasi pada hasil yang relevan dengan RQ dan kontribusi artikel.


In [ ]:

from __future__ import annotations

import json
import platform
import shutil
import sys
import textwrap
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

print('Python   :', sys.version.split()[0])
print('Pandas   :', pd.__version__)
print('NumPy    :', np.__version__)
print('Platform :', platform.platform())


In [ ]:

CURRENT_DIR = Path.cwd().resolve()
WORKBOOK_NAME = 'Dataset_Gabungan_Kemdiktisaintek_2025.xlsx'


def candidate_roots(start: Path) -> list[Path]:
    candidates: list[Path] = []

    def add(path: Path) -> None:
        try:
            path = path.resolve()
        except OSError:
            return
        if path not in candidates:
            candidates.append(path)

    add(start)
    for parent in start.parents:
        add(parent)
    try:
        for child in start.iterdir():
            if child.is_dir():
                add(child)
    except (PermissionError, FileNotFoundError):
        pass

    for base in list(candidates):
        for name in [
            'pt_education_research_project',
            'pt_education_research_project_00_01_02_03_04_05',
            'pt_education_research_project_00_01_02_03_04_05_06',
        ]:
            add(base / name)
    return candidates


PROJECT_ROOT = None
for candidate in candidate_roots(CURRENT_DIR):
    if (
        (candidate / 'outputs' / 'tables' / '05_readiness_checklist.csv').exists()
        and (candidate / 'data' / 'processed' / WORKBOOK_NAME).exists()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Folder proyek lengkap tidak ditemukan. Ekstrak paket proyek dan '
        'jalankan Jupyter dari folder proyek.'
    )

TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
ARTICLE_DIR = PROJECT_ROOT / 'outputs' / 'article_ready'
ARTICLE_TABLE_DIR = ARTICLE_DIR / 'tables'
ARTICLE_FIGURE_DIR = ARTICLE_DIR / 'figures'
ARTICLE_SUPP_DIR = ARTICLE_DIR / 'supplementary'
WORKBOOK_PATH = PROJECT_ROOT / 'data' / 'processed' / WORKBOOK_NAME

for directory in [ARTICLE_TABLE_DIR, ARTICLE_FIGURE_DIR, ARTICLE_SUPP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARTICLE_DIR :', ARTICLE_DIR)



## 1. Audit prasyarat dan inventaris hasil

Notebook hanya dilanjutkan apabila Notebook 01–05 seluruhnya berstatus `PASS`
dan semua sumber tabel utama tersedia.


In [ ]:

prerequisite_rows = []
for notebook_number in range(1, 6):
    path = TABLE_DIR / f'{notebook_number:02d}_readiness_checklist.csv'
    if not path.exists():
        prerequisite_rows.append({
            'notebook': f'{notebook_number:02d}',
            'file_exists': False,
            'checks': 0,
            'pass_checks': 0,
            'status': 'FAIL',
        })
        continue
    checklist = pd.read_csv(path)
    all_pass = checklist['status'].eq('PASS').all()
    prerequisite_rows.append({
        'notebook': f'{notebook_number:02d}',
        'file_exists': True,
        'checks': len(checklist),
        'pass_checks': int(checklist['status'].eq('PASS').sum()),
        'status': 'PASS' if all_pass else 'FAIL',
    })

prerequisite_audit = pd.DataFrame(prerequisite_rows)
prerequisite_audit.to_csv(
    ARTICLE_TABLE_DIR / '06_prerequisite_audit.csv', index=False
)
display(prerequisite_audit)
assert prerequisite_audit['status'].eq('PASS').all(), 'Prasyarat belum PASS.'

required_files = [
    '01_data_coverage.csv',
    '01_descriptive_statistics_strata.csv',
    '01_inequality_indices.csv',
    '01_theil_decomposition.csv',
    '01_province_summary.csv',
    '01_kabupaten_summary.csv',
    '01_summary_by_gender.csv',
    '01_summary_by_level.csv',
    '02_model_comparison.csv',
    '02_primary_model_irr.csv',
    '02_likelihood_ratio_tests.csv',
    '03_groupcv_model_ranking.csv',
    '03_generalization_gap.csv',
    '03_groupcv_province_bootstrap_ci.csv',
    '04_national_category_summary.csv',
    '04_national_composition_by_level.csv',
    '04_binomial_primary_odds_ratios.csv',
    '04_nb_flow_primary_irr.csv',
    '04_nb_interaction_combined_effects.csv',
    '05_integrated_evidence_matrix.csv',
    '05_gender_term_stability.csv',
    '05_predictive_rank_stability.csv',
    '05_stem_binomial_term_stability.csv',
    '05_stem_flow_term_stability.csv',
]
missing = [name for name in required_files if not (TABLE_DIR / name).exists()]
assert not missing, f'Output wajib tidak ditemukan: {missing}'
print(f'PASS — {len(required_files)} output sumber tersedia.')


In [ ]:

def read_table(name: str) -> pd.DataFrame:
    return pd.read_csv(TABLE_DIR / name)


def lorenz_coordinates(values) -> tuple[np.ndarray, np.ndarray]:
    array = np.asarray(values, dtype=float)
    array = array[np.isfinite(array) & (array >= 0)]
    array = np.sort(array)
    if array.size == 0 or array.sum() <= 0:
        return np.array([0, 1]), np.array([0, 1])
    cumulative = np.concatenate([[0.0], np.cumsum(array) / array.sum()])
    population = np.linspace(0, 1, array.size + 1)
    return population, cumulative


def save_figure(fig, filename: str) -> Path:
    path = ARTICLE_FIGURE_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return path


def format_p(value) -> str:
    if pd.isna(value):
        return ''
    value = float(value)
    return '<0.001' if value < 0.001 else f'{value:.3f}'


def ratio_text(estimate, lower, upper, digits=3) -> str:
    return f'{estimate:.{digits}f} ({lower:.{digits}f}–{upper:.{digits}f})'


def write_markdown_table(frame: pd.DataFrame, path: Path, title: str, note: str = ''):
    try:
        body = frame.to_markdown(index=False)
    except ImportError:
        body = frame.to_csv(index=False)
    text = f'# {title}\n\n{body}\n'
    if note:
        text += f'\n**Note.** {note}\n'
    path.write_text(text, encoding='utf-8')



## 2. Table 1 — Data sources and analytical samples


In [ ]:

source_meta = pd.read_excel(
    WORKBOOK_PATH, sheet_name='Sumber_Data', engine='openpyxl'
).dropna(subset=['nama_file_asli']).copy()

source_roles = {
    'Mahasiswa Lulus (Tanpa Akreditasi).xlsx': 'Outcome: graduate counts',
    'Mahasiswa Terdaftar.xlsx': 'Exposure: registered students',
    'Mahasiswa Aktif.xlsx': 'Student participation and MBKM',
    'Data Mahasiswa Baru Tahun 2025.xlsx': 'Entry composition and STEM flow exposure',
    'Perguruan Tinggi Terakreditasi.xlsx': 'Institutional capacity and accreditation',
    'data set jumlah dosen berdasarkan jabatan akademik.xls': 'Faculty rank and seniority',
    'data set jumlah dosen berdasarkan status sertifikasi.xls': 'Faculty certification',
    'data set jumlah dosen doktor bidang stem dan non stem.xls': 'Doctoral faculty STEM capacity',
}

T1A_sources = source_meta.loc[
    source_meta['status_penggunaan'].eq('Digunakan'),
    ['nama_file_asli', 'tanggal_terbit_portal', 'produsen_data',
     'jumlah_baris_data', 'url_halaman_dataset']
].copy()
T1A_sources.insert(
    1,
    'analytical_role',
    T1A_sources['nama_file_asli'].map(source_roles),
)
T1A_sources = T1A_sources.rename(columns={
    'nama_file_asli': 'source_file',
    'tanggal_terbit_portal': 'portal_publication_date',
    'produsen_data': 'data_producer',
    'jumlah_baris_data': 'source_rows',
    'url_halaman_dataset': 'source_url',
})

coverage = read_table('01_data_coverage.csv')
T1B_samples = coverage.copy()
T1B_samples['unit_of_analysis'] = T1B_samples['dataset'].map({
    'AD_Gender_Jenjang': 'District/city × gender × education level',
    'AD_STEM_Jenjang': 'District/city × STEM category × education level',
})
T1B_samples['denominator_coverage_pct'] = (
    T1B_samples['denominator_positive_n'] / T1B_samples['rows'] * 100
)
T1B_samples = T1B_samples[[
    'dataset', 'unit_of_analysis', 'rows', 'provinces', 'districts',
    'total_graduates', 'denominator_positive_n', 'denominator_coverage_pct',
    'flag_match_terdaftar_match_pct', 'flag_match_mahasiswa_baru_match_pct',
    'flag_match_pt_match_pct', 'flag_match_dosen_match_pct',
    'flag_match_dosen_doktor_match_pct',
]]

T1A_sources.to_csv(ARTICLE_TABLE_DIR / 'Table_1A_Data_Sources.csv', index=False)
T1B_samples.to_csv(ARTICLE_TABLE_DIR / 'Table_1B_Analytical_Samples.csv', index=False)
write_markdown_table(
    T1B_samples.round(3),
    ARTICLE_TABLE_DIR / 'Table_1B_Analytical_Samples.md',
    'Table 1. Analytical samples',
    'Coverage percentages refer to positive denominators. Graduate totals differ by three records because the gender dataset excludes the unknown-gender category.',
)
display(T1B_samples)



## 3. Table 2 — Descriptive statistics

Variabel dipilih berdasarkan model utama dan kontribusi artikel. Statistik penuh
Notebook 01 tetap tersedia sebagai tabel suplemen.


In [ ]:

descriptive = read_table('01_descriptive_statistics_strata.csv')
selected_variables = {
    'AD_Gender_Jenjang': [
        'jumlah_lulusan', 'jumlah_terdaftar', 'graduate_output_intensity',
        'proporsi_mbkm', 'jumlah_pt', 'proporsi_pt_akreditasi_tinggi',
        'proporsi_profesor', 'proporsi_dosen_tersertifikasi',
    ],
    'AD_STEM_Jenjang': [
        'jumlah_lulusan', 'jumlah_mahasiswa_baru',
        'graduate_to_entry_flow_ratio', 'jumlah_pt',
        'proporsi_pt_akreditasi_tinggi', 'dosen_doktor_sesuai_bidang',
        'proporsi_doktor_stem',
    ],
}

mask = pd.Series(False, index=descriptive.index)
for dataset, variables in selected_variables.items():
    mask |= descriptive['dataset'].eq(dataset) & descriptive['variable'].isin(variables)

T2 = descriptive.loc[mask, [
    'dataset', 'variable', 'n', 'missing_n', 'missing_pct', 'mean', 'std',
    'median', 'q1', 'q3', 'min', 'max', 'skewness', 'variance_to_mean',
]].copy()
T2['iqr'] = T2['q3'] - T2['q1']
T2 = T2[[
    'dataset', 'variable', 'n', 'missing_n', 'missing_pct', 'mean', 'std',
    'median', 'q1', 'q3', 'iqr', 'min', 'max', 'skewness', 'variance_to_mean',
]]
T2.to_csv(ARTICLE_TABLE_DIR / 'Table_2_Descriptive_Statistics.csv', index=False)
write_markdown_table(
    T2.round(4),
    ARTICLE_TABLE_DIR / 'Table_2_Descriptive_Statistics.md',
    'Table 2. Descriptive statistics for selected analytical variables',
    'Graduate-output and graduate-entry flow ratios are cross-sectional flow-to-stock measures, not cohort completion rates.',
)
display(T2)



## 4. Table 3 — Regional inequality and Theil decomposition


In [ ]:

inequality = read_table('01_inequality_indices.csv')
theil = read_table('01_theil_decomposition.csv')

selected_indicators = [
    'Provinsi — Jumlah lulusan',
    'Kabupaten/Kota — Jumlah lulusan',
    'Provinsi — Intensitas keluaran lulusan — tidak dibobot',
    'Kabupaten/Kota — Intensitas keluaran lulusan — tidak dibobot',
]
T3A = inequality.loc[
    inequality['indicator'].isin(selected_indicators),
    ['indicator', 'n_units', 'total', 'mean', 'median', 'gini', 'gini_lower',
     'gini_upper', 'theil_t', 'theil_lower', 'theil_upper', 'top_5_share',
     'top_10_share', 'interpretation_note']
].copy()

T3B = theil.copy()
T3B[['between_share', 'within_share']] *= 100
T3B = T3B.rename(columns={
    'between_share': 'between_province_share_pct',
    'within_share': 'within_province_share_pct',
})

T3A.to_csv(ARTICLE_TABLE_DIR / 'Table_3A_Regional_Inequality.csv', index=False)
T3B.to_csv(ARTICLE_TABLE_DIR / 'Table_3B_Theil_Decomposition.csv', index=False)
write_markdown_table(
    T3A.round(4),
    ARTICLE_TABLE_DIR / 'Table_3A_Regional_Inequality.md',
    'Table 3. Regional inequality in graduate production',
    'Bootstrap 95% confidence intervals are shown for Gini and Theil T. Ratios are unweighted across geographical units.',
)
display(T3A)
display(T3B)



## 5. Table 4 — Negative Binomial production function


In [ ]:

model_comparison = read_table('02_model_comparison.csv').copy()
T4A = model_comparison[[
    'model', 'nobs', 'df_model', 'log_likelihood', 'aic', 'bic',
    'mcfadden_pseudo_r2', 'alpha', 'delta_aic', 'aic_weight', 'converged',
]].copy()
T4A = T4A.sort_values('aic')

irr = read_table('02_primary_model_irr.csv').copy()
T4B = irr.loc[
    ~irr['term'].isin(['Intercept', 'alpha']),
    ['term_label', 'term_type', 'coefficient', 'std_error', 'irr',
     'irr_ci_lower', 'irr_ci_upper', 'p_value', 'significant_0_05']
].copy()
T4B['p_value_display'] = T4B['p_value'].map(format_p)
T4B['irr_95_ci'] = T4B.apply(
    lambda row: ratio_text(row['irr'], row['irr_ci_lower'], row['irr_ci_upper']), axis=1
)

T4A.to_csv(ARTICLE_TABLE_DIR / 'Table_4A_NB_Model_Comparison.csv', index=False)
T4B.to_csv(ARTICLE_TABLE_DIR / 'Table_4B_Main_Model_IRR.csv', index=False)
write_markdown_table(
    T4B[['term_label', 'irr_95_ci', 'p_value_display']].copy(),
    ARTICLE_TABLE_DIR / 'Table_4B_Main_Model_IRR.md',
    'Table 4. Incidence rate ratios from the primary Negative Binomial model',
    'Continuous predictors are standardized. Standard errors are clustered by province. S1 and male strata are reference categories.',
)
display(T4A)
display(T4B[['term_label', 'irr_95_ci', 'p_value_display']])



## 6. Table 5 — Out-of-province predictive performance


In [ ]:

ranking = read_table('03_groupcv_model_ranking.csv')
gaps = read_table('03_generalization_gap.csv')
bootstrap_ci = read_table('03_groupcv_province_bootstrap_ci.csv')

rmsle_ci = bootstrap_ci.loc[bootstrap_ci['metric'].eq('rmsle'), [
    'model', 'ci_lower_2_5', 'ci_upper_97_5'
]].rename(columns={
    'ci_lower_2_5': 'rmsle_ci_lower',
    'ci_upper_97_5': 'rmsle_ci_upper',
})

T5 = (
    ranking.merge(gaps, on='model', how='left')
    .merge(rmsle_ci, on='model', how='left')
)
T5 = T5[[
    'rank_rmsle', 'model', 'rmsle', 'rmsle_ci_lower', 'rmsle_ci_upper',
    'mae', 'log_r2', 'predicted_to_observed_ratio',
    'rmsle__RandomKFold', 'rmsle_gap_group_minus_random',
    'rmsle_ratio_group_over_random',
]].sort_values('rank_rmsle')
T5['group_rmsle_95_ci'] = T5.apply(
    lambda row: ratio_text(row['rmsle'], row['rmsle_ci_lower'], row['rmsle_ci_upper']), axis=1
)

T5.to_csv(ARTICLE_TABLE_DIR / 'Table_5_Out_of_Province_Performance.csv', index=False)
write_markdown_table(
    T5[['rank_rmsle', 'model', 'group_rmsle_95_ci', 'mae', 'log_r2',
        'predicted_to_observed_ratio', 'rmsle_gap_group_minus_random']].round(4),
    ARTICLE_TABLE_DIR / 'Table_5_Out_of_Province_Performance.md',
    'Table 5. Province-disjoint predictive performance',
    'RMSLE confidence intervals use province-level bootstrap resampling. Lower RMSLE and MAE are better; calibration ratio equals predicted total divided by observed total.',
)
display(T5)



## 7. Table 6 — STEM composition and graduate-entry flow


In [ ]:

national_stem = read_table('04_national_category_summary.csv')
composition_by_level = read_table('04_national_composition_by_level.csv')
binomial_or = read_table('04_binomial_primary_odds_ratios.csv')
flow_irr = read_table('04_nb_flow_primary_irr.csv')
combined = read_table('04_nb_interaction_combined_effects.csv')

T6A = national_stem.copy()
T6A[['proporsi_lulusan', 'proporsi_mahasiswa_baru']] *= 100
T6A = T6A.rename(columns={
    'proporsi_lulusan': 'graduate_share_pct',
    'proporsi_mahasiswa_baru': 'new_student_share_pct',
})
T6A['composition_gap_pp'] = (
    T6A['graduate_share_pct'] - T6A['new_student_share_pct']
)

key_binomial_terms = {
    'z_stem_new_share': 'STEM entry share (per 1 SD)',
    'z_stem_new_prodi_high_share': 'STEM students in highly accredited programs (per 1 SD)',
    'z_doctor_stem_share': 'STEM doctoral-faculty share (per 1 SD)',
}
T6B = binomial_or.loc[
    binomial_or['term'].isin(key_binomial_terms),
    ['term', 'odds_ratio', 'odds_ratio_ci_lower', 'odds_ratio_ci_upper', 'p_value']
].copy()
T6B['outcome'] = 'STEM graduate composition'
T6B['effect_measure'] = 'Odds ratio'
T6B['predictor'] = T6B['term'].map(key_binomial_terms)
T6B = T6B.rename(columns={
    'odds_ratio': 'estimate',
    'odds_ratio_ci_lower': 'ci_lower',
    'odds_ratio_ci_upper': 'ci_upper',
})

stem_term = 'C(a_stem, Treatment(reference="NON STEM"))[T.STEM]'
key_flow_terms = {
    stem_term: 'STEM vs non-STEM at mean regional capacity',
    'z_cert_share': 'Certified-faculty share (per 1 SD)',
    'z_pt_high_share': 'High-accreditation institution share (per 1 SD)',
}
T6C = flow_irr.loc[
    flow_irr['term'].isin(key_flow_terms),
    ['term', 'irr', 'irr_ci_lower', 'irr_ci_upper', 'p_value']
].copy()
T6C['outcome'] = 'Graduate-entry flow intensity'
T6C['effect_measure'] = 'Incidence rate ratio'
T6C['predictor'] = T6C['term'].map(key_flow_terms)
T6C = T6C.rename(columns={
    'irr': 'estimate', 'irr_ci_lower': 'ci_lower', 'irr_ci_upper': 'ci_upper'
})

T6_effects = pd.concat([
    T6B[['outcome', 'predictor', 'effect_measure', 'estimate', 'ci_lower', 'ci_upper', 'p_value']],
    T6C[['outcome', 'predictor', 'effect_measure', 'estimate', 'ci_lower', 'ci_upper', 'p_value']],
], ignore_index=True)
T6_effects['estimate_95_ci'] = T6_effects.apply(
    lambda row: ratio_text(row['estimate'], row['ci_lower'], row['ci_upper']), axis=1
)
T6_effects['p_value_display'] = T6_effects['p_value'].map(format_p)

T6A.to_csv(ARTICLE_TABLE_DIR / 'Table_6A_National_STEM_Composition.csv', index=False)
composition_by_level.to_csv(
    ARTICLE_TABLE_DIR / 'Table_6B_STEM_Composition_by_Level.csv', index=False
)
T6_effects.to_csv(ARTICLE_TABLE_DIR / 'Table_6C_STEM_Model_Effects.csv', index=False)
combined.to_csv(ARTICLE_TABLE_DIR / 'Table_6D_STEM_Combined_Effects.csv', index=False)
write_markdown_table(
    T6_effects[['outcome', 'predictor', 'effect_measure', 'estimate_95_ci', 'p_value_display']],
    ARTICLE_TABLE_DIR / 'Table_6C_STEM_Model_Effects.md',
    'Table 6. Principal STEM composition and graduate-entry flow results',
    'The flow outcome is not a cohort completion rate. Continuous predictors are standardized.',
)
display(T6A)
display(T6_effects)



## 8. Table 7 — Robustness and sensitivity evidence


In [ ]:

summary_05 = json.loads((TABLE_DIR / '05_summary.json').read_text(encoding='utf-8'))
evidence = read_table('05_integrated_evidence_matrix.csv')
gender_stability = read_table('05_gender_term_stability.csv')
stem_bin_stability = read_table('05_stem_binomial_term_stability.csv')
stem_flow_stability = read_table('05_stem_flow_term_stability.csv')
predictive_stability = read_table('05_predictive_rank_stability.csv')

female_range = gender_stability.loc[
    gender_stability['concept'].eq('female_vs_male')
].iloc[0]
active_range = gender_stability.loc[
    gender_stability['concept'].eq('active_share_per_sd')
].iloc[0]
stem_entry_range = stem_bin_stability.loc[
    stem_bin_stability['concept'].eq('stem_entry_share_per_sd')
].iloc[0]
stem_flow_range = stem_flow_stability.loc[
    stem_flow_stability['concept'].eq('stem_vs_nonstem_at_mean_capacity')
].iloc[0]

robustness_rows = [
    {
        'conclusion': 'Female strata have higher graduate output intensity',
        'primary_estimate': summary_05['female_primary_irr'],
        'effect_measure': 'IRR',
        'specification_min': female_range['ratio_minimum'],
        'specification_max': female_range['ratio_maximum'],
        'specification_direction_pct': summary_05['female_specification_positive_pct'],
        'leave_one_province_direction_pct': summary_05['female_loo_positive_pct'],
        'bootstrap_ci_lower': summary_05['female_bootstrap_ci_lower'],
        'bootstrap_ci_upper': summary_05['female_bootstrap_ci_upper'],
        'robustness_grade': 'Strong',
    },
    {
        'conclusion': 'Active-student share is positively associated with graduate output intensity',
        'primary_estimate': summary_05['active_primary_irr'],
        'effect_measure': 'IRR per SD',
        'specification_min': active_range['ratio_minimum'],
        'specification_max': active_range['ratio_maximum'],
        'specification_direction_pct': summary_05['active_specification_positive_pct'],
        'leave_one_province_direction_pct': summary_05['active_loo_positive_pct'],
        'bootstrap_ci_lower': np.nan,
        'bootstrap_ci_upper': np.nan,
        'robustness_grade': 'Strong',
    },
    {
        'conclusion': 'STEM entry composition predicts STEM graduate composition',
        'primary_estimate': summary_05['stem_entry_primary_or'],
        'effect_measure': 'OR per SD',
        'specification_min': stem_entry_range['ratio_minimum'],
        'specification_max': stem_entry_range['ratio_maximum'],
        'specification_direction_pct': summary_05['stem_entry_specification_positive_pct'],
        'leave_one_province_direction_pct': summary_05['stem_entry_loo_positive_pct'],
        'bootstrap_ci_lower': np.nan,
        'bootstrap_ci_upper': np.nan,
        'robustness_grade': 'Strong',
    },
    {
        'conclusion': 'STEM graduate-entry flow intensity is lower than non-STEM',
        'primary_estimate': summary_05['stem_flow_primary_mean_capacity_irr'],
        'effect_measure': 'IRR',
        'specification_min': stem_flow_range['ratio_minimum'],
        'specification_max': stem_flow_range['ratio_maximum'],
        'specification_direction_pct': summary_05['stem_flow_specification_negative_pct'],
        'leave_one_province_direction_pct': summary_05['stem_flow_loo_negative_pct'],
        'bootstrap_ci_lower': np.nan,
        'bootstrap_ci_upper': np.nan,
        'robustness_grade': 'Strong',
    },
    {
        'conclusion': 'One predictive model dominates all province-volume scenarios',
        'primary_estimate': summary_05['predictive_best_model_wins'] / summary_05['predictive_scenarios'],
        'effect_measure': 'Scenario-win proportion',
        'specification_min': np.nan,
        'specification_max': np.nan,
        'specification_direction_pct': 100 * summary_05['predictive_best_model_wins'] / summary_05['predictive_scenarios'],
        'leave_one_province_direction_pct': np.nan,
        'bootstrap_ci_lower': np.nan,
        'bootstrap_ci_upper': np.nan,
        'robustness_grade': 'Sensitive',
    },
]
T7 = pd.DataFrame(robustness_rows)
T7.to_csv(ARTICLE_TABLE_DIR / 'Table_7_Robustness_Evidence.csv', index=False)
evidence.to_csv(ARTICLE_TABLE_DIR / 'Supplementary_Integrated_Evidence_Matrix.csv', index=False)
predictive_stability.to_csv(
    ARTICLE_TABLE_DIR / 'Supplementary_Predictive_Rank_Stability.csv', index=False
)
write_markdown_table(
    T7.round(4),
    ARTICLE_TABLE_DIR / 'Table_7_Robustness_Evidence.md',
    'Table 7. Robustness and sensitivity of principal conclusions',
    'Direction percentages quantify consistency across prespecified specifications or leave-one-province-out refits.',
)
display(T7)



## 9. Results ledger and manuscript-safe terminology

Ledger ini menjadi sumber tunggal angka yang akan digunakan saat menulis Results,
Discussion, Abstract, dan Conclusion.


In [ ]:

summary_01 = json.loads((TABLE_DIR / '01_descriptive_summary.json').read_text(encoding='utf-8'))
summary_02 = json.loads((TABLE_DIR / '02_summary.json').read_text(encoding='utf-8'))
summary_03 = json.loads((TABLE_DIR / '03_summary.json').read_text(encoding='utf-8'))
summary_04 = json.loads((TABLE_DIR / '04_summary.json').read_text(encoding='utf-8'))

results_ledger = pd.DataFrame([
    {
        'claim_id': 'C01',
        'section': 'Results—Descriptive',
        'claim': 'Graduate production is highly concentrated geographically.',
        'estimate': summary_01['district_gini_graduates'],
        'ci_or_range': 'See Table 3 bootstrap CI',
        'source': '01_inequality_indices.csv',
        'approved_wording': 'The district/city Gini coefficient for graduate counts was 0.823, indicating pronounced geographic concentration.',
        'avoid_wording': 'Regional inequality caused graduate outcomes.',
    },
    {
        'claim_id': 'C02',
        'section': 'Results—Descriptive',
        'claim': 'Most Theil inequality occurs within provinces.',
        'estimate': float(T3B.loc[T3B['variable'].eq('jumlah_lulusan'), 'within_province_share_pct'].iloc[0]),
        'ci_or_range': 'Not applicable',
        'source': '01_theil_decomposition.csv',
        'approved_wording': 'Approximately 81.5% of Theil inequality in graduate counts occurred within rather than between provinces.',
        'avoid_wording': 'Provincial policy explains 81.5% of inequality.',
    },
    {
        'claim_id': 'C03',
        'section': 'Results—Negative Binomial',
        'claim': 'Female strata have higher graduate output intensity.',
        'estimate': summary_02['female_vs_male_irr'],
        'ci_or_range': ratio_text(summary_02['female_vs_male_irr'], summary_02['female_vs_male_ci_lower'], summary_02['female_vs_male_ci_upper']),
        'source': '02_primary_model_irr.csv',
        'approved_wording': 'Female strata had 9.4% higher expected graduate output intensity than male strata, conditional on the model covariates.',
        'avoid_wording': 'Female students are 9.4% more likely to graduate.',
    },
    {
        'claim_id': 'C04',
        'section': 'Results—Negative Binomial',
        'claim': 'Active-student share is positively associated with output intensity.',
        'estimate': summary_02['active_share_irr_per_sd'],
        'ci_or_range': ratio_text(summary_02['active_share_irr_per_sd'], summary_02['active_share_ci_lower'], summary_02['active_share_ci_upper']),
        'source': '02_primary_model_irr.csv',
        'approved_wording': 'A one-standard-deviation increase in active-student share was associated with a 19.4% increase in expected graduate output intensity.',
        'avoid_wording': 'Increasing active students will cause 19.4% more graduates.',
    },
    {
        'claim_id': 'C05',
        'section': 'Results—Prediction',
        'claim': 'TweedieRate achieved the best mean province-disjoint RMSLE.',
        'estimate': summary_03['groupcv_province_rmsle'],
        'ci_or_range': ratio_text(summary_03['groupcv_province_rmsle'], summary_03['groupcv_province_rmsle_ci_lower'], summary_03['groupcv_province_rmsle_ci_upper']),
        'source': '03_groupcv_model_ranking.csv',
        'approved_wording': 'TweedieRate ranked first by province-level GroupKFold RMSLE, although its dominance was sensitive to province-volume scenarios.',
        'avoid_wording': 'TweedieRate is universally the best model.',
    },
    {
        'claim_id': 'C06',
        'section': 'Results—STEM',
        'claim': 'STEM entry composition predicts STEM graduate composition.',
        'estimate': summary_04['stem_new_share_odds_ratio_per_sd'],
        'ci_or_range': ratio_text(summary_04['stem_new_share_odds_ratio_per_sd'], summary_04['stem_new_share_or_ci_lower'], summary_04['stem_new_share_or_ci_upper']),
        'source': '04_binomial_primary_odds_ratios.csv',
        'approved_wording': 'A one-standard-deviation increase in STEM entry share was associated with approximately threefold higher odds of STEM graduate composition.',
        'avoid_wording': 'STEM recruitment triples individual graduation probability.',
    },
    {
        'claim_id': 'C07',
        'section': 'Results—STEM',
        'claim': 'STEM graduate-entry flow intensity is lower than non-STEM.',
        'estimate': summary_04['stem_vs_nonstem_flow_irr'],
        'ci_or_range': ratio_text(summary_04['stem_vs_nonstem_flow_irr'], summary_04['stem_vs_nonstem_flow_ci_lower'], summary_04['stem_vs_nonstem_flow_ci_upper']),
        'source': '04_nb_flow_primary_irr.csv',
        'approved_wording': 'At mean regional capacity, STEM graduate-entry flow intensity was approximately 30.2% lower than non-STEM.',
        'avoid_wording': 'STEM students have a 30.2% lower completion rate.',
    },
])

results_ledger.to_csv(ARTICLE_TABLE_DIR / '06_Results_Ledger.csv', index=False)
write_markdown_table(
    results_ledger[['claim_id', 'section', 'approved_wording', 'source']],
    ARTICLE_TABLE_DIR / '06_Results_Ledger.md',
    'Results ledger',
    'Use approved wording consistently across the manuscript. The avoid-wording field documents ecological and causal interpretation risks.',
)
display(results_ledger)



## 10. Figure 1 — Integrated data and analytical workflow


In [ ]:

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 7.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

boxes = [
    (0.4, 5.5, 3.0, 1.5, 'Administrative inputs',
     'Graduates, registered,\nactive and new students'),
    (0.4, 3.0, 3.0, 1.5, 'Institutional inputs',
     'Institutions, accreditation,\nand faculty quality'),
    (4.2, 4.25, 3.2, 1.7, 'Data integration and audit',
     'Harmonized geography\nUnique analytical keys\nMissing/unmatched audit'),
    (8.2, 5.5, 2.6, 1.5, 'Dataset A',
     'Gender × level × district'),
    (8.2, 3.0, 2.6, 1.5, 'Dataset B',
     'STEM × level × district'),
    (11.5, 5.5, 2.1, 1.5, 'Inference',
     'NB2 + clustered SE'),
    (11.5, 3.0, 2.1, 1.5, 'Prediction',
     'Province-disjoint CV'),
    (5.2, 0.45, 3.8, 1.35, 'Robustness and article-ready synthesis',
     'Specification grid, LOO, bootstrap,\ntables and figures'),
]

for x, y, w, h, title, subtitle in boxes:
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.03,rounding_size=0.12',
        linewidth=1.2,
        facecolor='white',
        edgecolor='black',
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h*0.66, title, ha='center', va='center',
            fontsize=11, fontweight='bold')
    ax.text(x + w/2, y + h*0.30, subtitle, ha='center', va='center',
            fontsize=8.5, linespacing=1.25)

arrows = [
    ((3.4, 6.25), (4.2, 5.35)),
    ((3.4, 3.75), (4.2, 4.85)),
    ((7.4, 5.2), (8.2, 6.25)),
    ((7.4, 4.9), (8.2, 3.75)),
    ((10.8, 6.25), (11.5, 6.25)),
    ((10.8, 3.75), (11.5, 3.75)),
    ((12.55, 5.5), (8.3, 1.8)),
    ((12.55, 3.0), (8.8, 1.8)),
]
for start, end in arrows:
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=14,
                                 linewidth=1.1, connectionstyle='arc3,rad=0.0'))

ax.set_title('Integrated Administrative Data and Analytical Workflow',
             fontsize=15, fontweight='bold', pad=16)
save_figure(fig, 'Figure_1_Data_Integration_and_Analysis_Workflow.png')



## 11. Figure 2 — Lorenz curves for graduate counts


In [ ]:

province = read_table('01_province_summary.csv')
district = read_table('01_kabupaten_summary.csv')

x_p, y_p = lorenz_coordinates(province['jumlah_lulusan'])
x_d, y_d = lorenz_coordinates(district['jumlah_lulusan'])

fig, ax = plt.subplots(figsize=(7.5, 7))
ax.plot([0, 1], [0, 1], linestyle='--', label='Equality line')
ax.plot(x_p, y_p, linewidth=2, label='Province (Gini = 0.572)')
ax.plot(x_d, y_d, linewidth=2, label='District/city (Gini = 0.823)')
ax.set_xlabel('Cumulative share of geographical units')
ax.set_ylabel('Cumulative share of graduates')
ax.set_title('Geographic Concentration of Higher-Education Graduates')
ax.legend()
ax.grid(alpha=0.2)
save_figure(fig, 'Figure_2_Lorenz_Curves_Graduate_Counts.png')



## 12. Figure 3 — Principal IRRs from the production-function model


In [ ]:

key_labels = [
    'Perempuan vs laki-laki',
    'Proporsi mahasiswa aktif (per 1 SD)',
    'Proporsi MBKM (per 1 SD)',
    'Log jumlah PT (per 1 SD)',
    'Proporsi PT akreditasi tinggi (per 1 SD)',
    'Proporsi profesor (per 1 SD)',
    'Proporsi dosen tersertifikasi (per 1 SD)',
]
forest = T4B.loc[T4B['term_label'].isin(key_labels)].copy()
forest['order'] = forest['term_label'].map({label: i for i, label in enumerate(key_labels)})
forest = forest.sort_values('order', ascending=False)

fig, ax = plt.subplots(figsize=(9, 6.5))
y = np.arange(len(forest))
xerr = np.vstack([
    forest['irr'] - forest['irr_ci_lower'],
    forest['irr_ci_upper'] - forest['irr'],
])
ax.errorbar(forest['irr'], y, xerr=xerr, fmt='o', capsize=3)
ax.axvline(1, linestyle='--', linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(forest['term_label'])
ax.set_xscale('log')
ax.set_xlabel('Incidence rate ratio (log scale)')
ax.set_title('Principal Associations with Graduate Output Intensity')
ax.grid(axis='x', alpha=0.2)
save_figure(fig, 'Figure_3_Principal_Negative_Binomial_IRRs.png')



## 13. Figure 4 — Province-disjoint predictive performance


In [ ]:

pred_plot = T5.sort_values('rmsle', ascending=False).copy()

fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(pred_plot))
xerr = np.vstack([
    pred_plot['rmsle'] - pred_plot['rmsle_ci_lower'],
    pred_plot['rmsle_ci_upper'] - pred_plot['rmsle'],
])
ax.errorbar(pred_plot['rmsle'], y, xerr=xerr, fmt='o', capsize=3)
ax.set_yticks(y)
ax.set_yticklabels(pred_plot['model'])
ax.set_xlabel('Province-level RMSLE (95% bootstrap CI; lower is better)')
ax.set_title('Out-of-Province Predictive Performance')
ax.grid(axis='x', alpha=0.2)
save_figure(fig, 'Figure_4_Out_of_Province_Predictive_Performance.png')



## 14. Figure 5 — Principal STEM composition and flow effects


In [ ]:

stem_plot = T6_effects.copy()
stem_plot['label'] = stem_plot['outcome'] + ': ' + stem_plot['predictor']
stem_plot = stem_plot.sort_values('estimate', ascending=False)

fig_height = max(6.5, 0.65 * len(stem_plot))
fig, ax = plt.subplots(figsize=(10, fig_height))
y = np.arange(len(stem_plot))
xerr = np.vstack([
    stem_plot['estimate'] - stem_plot['ci_lower'],
    stem_plot['ci_upper'] - stem_plot['estimate'],
])
ax.errorbar(stem_plot['estimate'], y, xerr=xerr, fmt='o', capsize=3)
ax.axvline(1, linestyle='--', linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(stem_plot['label'])
ax.set_xscale('log')
ax.set_xlabel('Odds ratio or incidence rate ratio (log scale)')
ax.set_title('Principal STEM Composition and Graduate-Entry Flow Effects')
ax.grid(axis='x', alpha=0.2)
save_figure(fig, 'Figure_5_Principal_STEM_Effects.png')



## 15. Figure 6 — Stability across specifications


In [ ]:

robust_plot = T7.iloc[:4].copy()
robust_plot['label'] = [
    'Female vs male',
    'Active-student share',
    'STEM entry composition',
    'STEM vs non-STEM flow',
]
robust_plot = robust_plot.sort_values('primary_estimate', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5.8))
y = np.arange(len(robust_plot))
xerr = np.vstack([
    robust_plot['primary_estimate'] - robust_plot['specification_min'],
    robust_plot['specification_max'] - robust_plot['primary_estimate'],
])
ax.errorbar(robust_plot['primary_estimate'], y, xerr=xerr, fmt='o', capsize=4)
ax.axvline(1, linestyle='--', linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(robust_plot['label'])
ax.set_xscale('log')
ax.set_xlabel('Primary ratio and range across specifications (log scale)')
ax.set_title('Robustness of Principal Effect Estimates')
ax.grid(axis='x', alpha=0.2)
save_figure(fig, 'Figure_6_Robustness_Across_Specifications.png')



## 16. Captions, supplementary files, and article-ready index


In [ ]:

figure_captions = pd.DataFrame([
    {
        'figure_id': 'Figure 1',
        'filename': 'Figure_1_Data_Integration_and_Analysis_Workflow.png',
        'title': 'Integrated administrative data and analytical workflow',
        'caption': 'Eight administrative data sources were harmonized at common geographical and educational units to construct gender–level and STEM–level analytical datasets. The workflow separates inference, province-disjoint prediction, and robustness analysis.',
        'suggested_section': 'Methods',
    },
    {
        'figure_id': 'Figure 2',
        'filename': 'Figure_2_Lorenz_Curves_Graduate_Counts.png',
        'title': 'Geographic concentration of higher-education graduates',
        'caption': 'Lorenz curves compare the distribution of graduate counts across 34 provinces and 414 districts/cities. Greater curvature at district/city level indicates substantially stronger within-country concentration.',
        'suggested_section': 'Results—Regional inequality',
    },
    {
        'figure_id': 'Figure 3',
        'filename': 'Figure_3_Principal_Negative_Binomial_IRRs.png',
        'title': 'Principal associations with graduate output intensity',
        'caption': 'Incidence rate ratios and 95% cluster-robust confidence intervals from the primary Negative Binomial model. Continuous predictors are standardized; the vertical reference line denotes no association.',
        'suggested_section': 'Results—Production function',
    },
    {
        'figure_id': 'Figure 4',
        'filename': 'Figure_4_Out_of_Province_Predictive_Performance.png',
        'title': 'Out-of-province predictive performance',
        'caption': 'Province-level RMSLE and 95% province-bootstrap confidence intervals under GroupKFold validation. Lower RMSLE indicates better generalization to provinces excluded from model training.',
        'suggested_section': 'Results—Predictive validation',
    },
    {
        'figure_id': 'Figure 5',
        'filename': 'Figure_5_Principal_STEM_Effects.png',
        'title': 'Principal STEM composition and graduate-entry flow effects',
        'caption': 'Odds ratios for STEM graduate composition and incidence rate ratios for graduate-entry flow intensity. Ratios are cross-sectional ecological associations and do not represent individual completion probabilities.',
        'suggested_section': 'Results—STEM analysis',
    },
    {
        'figure_id': 'Figure 6',
        'filename': 'Figure_6_Robustness_Across_Specifications.png',
        'title': 'Robustness of principal effect estimates',
        'caption': 'Primary ratios and their observed ranges across prespecified alternative model specifications. All four substantive conclusions retained their expected direction across specifications.',
        'suggested_section': 'Results—Robustness',
    },
])
figure_captions.to_csv(ARTICLE_TABLE_DIR / '06_Figure_Captions.csv', index=False)

# Copy selected existing diagnostic figures as supplementary material.
supplementary_map = {
    '02_distribution_graduates_log1p.png': 'Figure_S1_Graduate_Count_Distribution_Log.png',
    '02_06_calibration_deciles.png': 'Figure_S2_NB_In_Sample_Calibration.png',
    '03_02_generalization_gap.png': 'Figure_S3_Random_vs_Group_Generalization_Gap.png',
    '03_05_best_province_residuals.png': 'Figure_S4_Best_Model_Province_Residuals.png',
    '04_01_stem_share_by_level.png': 'Figure_S5_STEM_Share_by_Education_Level.png',
    '05_04_gender_cluster_bootstrap_irr.png': 'Figure_S6_Gender_Cluster_Bootstrap.png',
}
for source_name, target_name in supplementary_map.items():
    source = FIGURE_DIR / source_name
    if source.exists():
        shutil.copy2(source, ARTICLE_SUPP_DIR / target_name)

# Full analysis tables retained as supplementary CSVs.
supplementary_tables = {
    '01_descriptive_statistics_strata.csv': 'Table_S1_Full_Descriptive_Statistics.csv',
    '02_primary_model_irr.csv': 'Table_S2_Full_Primary_NB_IRR.csv',
    '03_model_metrics.csv': 'Table_S3_Full_Predictive_Metrics.csv',
    '04_national_composition_by_level.csv': 'Table_S4_STEM_Composition_by_Level.csv',
    '05_gender_specification_coefficients.csv': 'Table_S5_Gender_Specification_Coefficients.csv',
    '05_stem_binomial_coefficients.csv': 'Table_S6_STEM_Composition_Specifications.csv',
    '05_stem_flow_coefficients.csv': 'Table_S7_STEM_Flow_Specifications.csv',
}
for source_name, target_name in supplementary_tables.items():
    source = TABLE_DIR / source_name
    if source.exists():
        shutil.copy2(source, ARTICLE_SUPP_DIR / target_name)

article_index = textwrap.dedent('''
# Article-ready output index

## Main tables

- Table 1A: Data sources
- Table 1B: Analytical samples
- Table 2: Descriptive statistics
- Table 3A: Regional inequality
- Table 3B: Theil decomposition
- Table 4A: Negative Binomial model comparison
- Table 4B: Main-model IRRs
- Table 5: Province-disjoint predictive performance
- Table 6A–D: STEM composition and flow
- Table 7: Robustness evidence

## Main figures

1. Integrated data and analytical workflow
2. Lorenz curves for graduate counts
3. Principal Negative Binomial IRRs
4. Out-of-province predictive performance
5. Principal STEM effects
6. Robustness across specifications

## Reporting sequence

Use Table 1–3 for descriptive and inequality results, Table 4 for inferential
results, Table 5 for predictive validation, Table 6 for STEM analysis, and
Table 7 for robustness. The Results Ledger is the authoritative source for
all headline numbers and manuscript-safe wording.
''').strip()
(ARTICLE_DIR / '06_Article_Ready_Index.md').write_text(article_index, encoding='utf-8')

# Caption/notes for tables.
table_manifest = pd.DataFrame([
    ['Table 1A', 'Table_1A_Data_Sources.csv', 'Data sources and analytical roles', 'Methods'],
    ['Table 1B', 'Table_1B_Analytical_Samples.csv', 'Analytical samples and data coverage', 'Methods'],
    ['Table 2', 'Table_2_Descriptive_Statistics.csv', 'Descriptive statistics', 'Results'],
    ['Table 3A', 'Table_3A_Regional_Inequality.csv', 'Regional inequality indices', 'Results'],
    ['Table 3B', 'Table_3B_Theil_Decomposition.csv', 'Theil decomposition', 'Results'],
    ['Table 4A', 'Table_4A_NB_Model_Comparison.csv', 'Negative Binomial model comparison', 'Results'],
    ['Table 4B', 'Table_4B_Main_Model_IRR.csv', 'Primary-model incidence rate ratios', 'Results'],
    ['Table 5', 'Table_5_Out_of_Province_Performance.csv', 'Province-disjoint predictive performance', 'Results'],
    ['Table 6A', 'Table_6A_National_STEM_Composition.csv', 'National STEM composition', 'Results'],
    ['Table 6B', 'Table_6B_STEM_Composition_by_Level.csv', 'STEM composition by education level', 'Results/Supplement'],
    ['Table 6C', 'Table_6C_STEM_Model_Effects.csv', 'Principal STEM model effects', 'Results'],
    ['Table 6D', 'Table_6D_STEM_Combined_Effects.csv', 'Combined STEM interaction effects', 'Supplement'],
    ['Table 7', 'Table_7_Robustness_Evidence.csv', 'Robustness evidence', 'Results'],
], columns=['table_id', 'filename', 'title', 'suggested_section'])
table_manifest.to_csv(ARTICLE_TABLE_DIR / '06_Table_Manifest.csv', index=False)

display(table_manifest)
display(figure_captions)



## 17. Final readiness checklist and article-ready archive


In [ ]:

expected_main_tables = [
    'Table_1A_Data_Sources.csv',
    'Table_1B_Analytical_Samples.csv',
    'Table_2_Descriptive_Statistics.csv',
    'Table_3A_Regional_Inequality.csv',
    'Table_3B_Theil_Decomposition.csv',
    'Table_4A_NB_Model_Comparison.csv',
    'Table_4B_Main_Model_IRR.csv',
    'Table_5_Out_of_Province_Performance.csv',
    'Table_6A_National_STEM_Composition.csv',
    'Table_6B_STEM_Composition_by_Level.csv',
    'Table_6C_STEM_Model_Effects.csv',
    'Table_6D_STEM_Combined_Effects.csv',
    'Table_7_Robustness_Evidence.csv',
    '06_Results_Ledger.csv',
    '06_Figure_Captions.csv',
    '06_Table_Manifest.csv',
]
expected_main_figures = [
    'Figure_1_Data_Integration_and_Analysis_Workflow.png',
    'Figure_2_Lorenz_Curves_Graduate_Counts.png',
    'Figure_3_Principal_Negative_Binomial_IRRs.png',
    'Figure_4_Out_of_Province_Predictive_Performance.png',
    'Figure_5_Principal_STEM_Effects.png',
    'Figure_6_Robustness_Across_Specifications.png',
]

readiness_06 = pd.DataFrame([
    {
        'check': 'Notebook 01–05 prerequisites PASS',
        'status': 'PASS' if prerequisite_audit['status'].eq('PASS').all() else 'FAIL',
        'evidence': f"{int(prerequisite_audit['status'].eq('PASS').sum())}/5 notebooks",
    },
    {
        'check': 'Main article tables complete',
        'status': 'PASS' if all((ARTICLE_TABLE_DIR / f).exists() for f in expected_main_tables) else 'FAIL',
        'evidence': f"{sum((ARTICLE_TABLE_DIR / f).exists() for f in expected_main_tables)}/{len(expected_main_tables)} files",
    },
    {
        'check': 'Main article figures complete',
        'status': 'PASS' if all((ARTICLE_FIGURE_DIR / f).exists() for f in expected_main_figures) else 'FAIL',
        'evidence': f"{sum((ARTICLE_FIGURE_DIR / f).exists() for f in expected_main_figures)}/{len(expected_main_figures)} figures",
    },
    {
        'check': 'Results ledger complete',
        'status': 'PASS' if len(results_ledger) >= 7 else 'FAIL',
        'evidence': f'{len(results_ledger)} headline claims',
    },
    {
        'check': 'Ecological interpretation safeguards documented',
        'status': 'PASS' if results_ledger['avoid_wording'].notna().all() else 'FAIL',
        'evidence': 'Approved and prohibited wording supplied for every headline claim',
    },
    {
        'check': 'Figure captions and placements complete',
        'status': 'PASS' if len(figure_captions) == 6 else 'FAIL',
        'evidence': f'{len(figure_captions)} captions',
    },
])
readiness_06.to_csv(ARTICLE_TABLE_DIR / '06_readiness_checklist.csv', index=False)
display(readiness_06)
assert readiness_06['status'].eq('PASS').all(), 'Notebook 06 belum seluruhnya PASS.'

# Create an article-only archive from within the notebook.
archive_path = PROJECT_ROOT / 'outputs' / 'Article_Ready_Tables_and_Figures.zip'
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in ARTICLE_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(ARTICLE_DIR.parent))

manifest_rows = []
for path in sorted(ARTICLE_DIR.rglob('*')):
    if path.is_file():
        manifest_rows.append({
            'relative_path': str(path.relative_to(PROJECT_ROOT)),
            'size_bytes': path.stat().st_size,
            'extension': path.suffix.lower(),
        })
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(ARTICLE_TABLE_DIR / '06_output_manifest.csv', index=False)

summary = {
    'main_tables': len(expected_main_tables),
    'main_figures': len(expected_main_figures),
    'supplementary_files': len(list(ARTICLE_SUPP_DIR.glob('*'))),
    'headline_claims': len(results_ledger),
    'article_archive': str(archive_path.relative_to(PROJECT_ROOT)),
    'status': 'PASS',
}
(ARTICLE_TABLE_DIR / '06_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)

print('=' * 78)
print('NOTEBOOK 06 LOLOS.')
print('Tabel dan figur siap digunakan untuk penyusunan artikel.')
print('=' * 78)
